# Deutsch-Jozsa Algorithm

Determines whether a function f:{0,1}^n -> {0,1} is constant or balanced
in a single query. For n=2, one query suffices vs 2^(n-1)+1 classical queries.

In [ ]:
import cirq
import numpy as np

## Oracle Construction

- Constant: f(x) = 0 (no-op) or f(x) = 1 (X on output)
- Balanced: f(x) = x0 (CNOT from q0) or f(x) = x0 XOR x1 (two CNOTs)

In [ ]:
def constant_oracle_0(qubits):
    return []

def constant_oracle_1(qubits):
    return [cirq.X(qubits[-1])]

def balanced_oracle_id(qubits):
    return [cirq.CNOT(qubits[0], qubits[-1])]

def balanced_oracle_xor(qubits):
    return [cirq.CNOT(qubits[0], qubits[-1]), cirq.CNOT(qubits[1], qubits[-1])]

## Run Deutsch-Jozsa

Apply H to all qubits (output starts at |1\u27e9), run oracle, apply H to inputs.
If all input qubits measure |0\u27e9, the function is constant.

In [ ]:
n = 2
sim = cirq.Simulator()

oracles = [
    ("Constant f(x)=0", constant_oracle_0),
    ("Constant f(x)=1", constant_oracle_1),
    ("Balanced f(x)=x0", balanced_oracle_id),
    ("Balanced f(x)=x0 XOR x1", balanced_oracle_xor),
]

for name, oracle_fn in oracles:
    qubits = cirq.LineQubit.range(n + 1)
    circuit = cirq.Circuit(
        cirq.X(qubits[-1]),
        [cirq.H(q) for q in qubits],
        *oracle_fn(qubits),
        [cirq.H(q) for q in qubits[:n]],
    )
    sv = sim.simulate(circuit).final_state_vector
    measured_zero = all(abs(sv[i]) < 1e-6 for i in range(2**n))
    verdict = "CONSTANT" if measured_zero else "BALANCED"
    print(f"{name}: {verdict}")
    for i in range(2**n):
        print(f"  |{i:0{n}b}\u27e9: {sv[i]:.4f}")
    print()